In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

data_dir = r"./../data/raw"
files = os.listdir(data_dir)
files = [f for f in files if f != "placeholder"]
print(files)

['credit_card_balance.csv', 'installments_payments.csv', 'application_train.csv', 'previous_application.csv', '.~lock.installments_payments.csv#', '.~lock.bureau.csv#', 'bureau.csv', 'sample_submission.csv', 'application_test.csv', 'HomeCredit_columns_description.csv', 'bureau_balance.csv', 'POS_CASH_balance.csv']


# installments_payments

In [29]:
table_name = "installments_payments.csv"
raw_data = pd.read_csv(os.path.join(data_dir, table_name))
raw_data

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585
...,...,...,...,...,...,...,...,...
13605396,2186857,428057,0.0,66,-1624.0,NaN,67.500,NaN
13605397,1310347,414406,0.0,47,-1539.0,NaN,67.500,NaN
13605398,1308766,402199,0.0,43,-7.0,NaN,43737.435,NaN
13605399,1062206,409297,0.0,43,-1986.0,NaN,67.500,NaN


In [33]:
modif_df = raw_data.copy()
modif_df['payement_time_delai'] = modif_df['DAYS_ENTRY_PAYMENT'] - modif_df['DAYS_INSTALMENT']
modif_df.loc[modif_df["payement_time_delai"] < 0, "payement_time_delai"] = 0
modif_df["payment_amount_diff"] = modif_df["AMT_PAYMENT"] - modif_df["AMT_INSTALMENT"]
modif_df["flag_default_payement"] = modif_df["payement_time_delai"].isna()



new_df = modif_df.groupby("SK_ID_PREV").agg({
    "NUM_INSTALMENT_VERSION": ["max"],
    "flag_default_payement": ["sum"],
    "payement_time_delai": ["mean"],
    "payment_amount_diff": ["mean"],
    
})
print(new_df.shape)
new_df.columns = ["NUM_INSTALMENT_VERSION_MAX", "FLAG_DEFAULT_PAYEMENT", "PAYEMENT_DELAI_TIME_MEAN", "PAYMENET_AMOUNT_DIFF_MEAN"]
new_df.isna().sum(0)

(997752, 4)


NUM_INSTALMENT_VERSION_MAX     0
FLAG_DEFAULT_PAYEMENT          0
PAYEMENT_DELAI_TIME_MEAN      78
PAYMENET_AMOUNT_DIFF_MEAN     78
dtype: int64

# POS

In [3]:
table_name = "POS_CASH_balance.csv"
raw_data = pd.read_csv(os.path.join(data_dir, table_name))
raw_data



,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0
...,...,...,...,...,...,...,...,...
10001353,2448283,226558,-20,6.0,0.0,Active,843,0
10001354,1717234,141565,-19,12.0,0.0,Active,602,0
10001355,1283126,315695,-21,10.0,0.0,Active,609,0
10001356,1082516,450255,-22,12.0,0.0,Active,614,0


In [ ]:
new_df = raw_data.groupby("SK_ID_PREV").agg({
    "NUM_INSTALMENT_VERSION": ["max"],
    "flag_default_payement": ["sum"],
    "payement_time_delai": ["mean"],
    "payment_amount_diff": ["mean"],
    
})

In [5]:
raw_data.isna().sum(0)

SK_ID_PREV                   0
SK_ID_CURR                   0
MONTHS_BALANCE               0
CNT_INSTALMENT           26071
CNT_INSTALMENT_FUTURE    26087
NAME_CONTRACT_STATUS         0
SK_DPD                       0
SK_DPD_DEF                   0
dtype: int64

In [10]:
raw_data[raw_data["CNT_INSTALMENT_FUTURE"].isna()]["NAME_CONTRACT_STATUS"].value_counts()

NAME_CONTRACT_STATUS
Signed                   20291
Returned to the store     2962
Approved                  2794
Active                      26
Canceled                    12
XNA                          2
Name: count, dtype: int64

In [9]:
raw_data["NAME_CONTRACT_STATUS"].value_counts()

NAME_CONTRACT_STATUS
Active                   9151119
Completed                 744883
Signed                     87260
Demand                      7065
Returned to the store       5461
Approved                    4917
Amortized debt               636
Canceled                      15
XNA                            2
Name: count, dtype: int64

In [17]:
raw_data[(raw_data["CNT_INSTALMENT_FUTURE"] == 0) & (raw_data["SK_DPD"] == 0)]["NAME_CONTRACT_STATUS"].value_counts()


NAME_CONTRACT_STATUS
Completed         734733
Active            299566
Demand              3555
Signed               830
Approved              55
Amortized debt        44
Name: count, dtype: int64

- Mean DPD
- Severe delinquency (>90 days)
- Fully paid ratio
- Ever had Completed contract

In [19]:
pos = raw_data.copy()
# pos['payement_time_delai'] = pos['DAYS_ENTRY_PAYMENT'] - pos['DAYS_INSTALMENT']
# pos.loc[pos["payement_time_delai"] < 0, "payement_time_delai"] = 0
# pos["payment_amount_diff"] = pos["AMT_PAYMENT"] - pos["AMT_INSTALMENT"]
# pos["flag_default_payement"] = pos["payement_time_delai"].isna()

# pos.loc[pos["payement_time_delai"] > 90, "payement_time_delai"] = 1


# new_df = pos.groupby("SK_ID_PREV").agg({
#     "NUM_INSTALMENT_VERSION": ["max"],
#     "flag_default_payement": ["sum"],
#     "payement_time_delai": ["mean"],
#     "payment_amount_diff": ["mean"],
    
# })
# print(new_df.shape)
# new_df.columns = ["NUM_INSTALMENT_VERSION_MAX", "FLAG_DEFAULT_PAYEMENT", "PAYEMENT_DELAI_TIME_MEAN", "PAYMENET_AMOUNT_DIFF_MEAN"]
# new_df.isna().sum(0)

In [31]:
pos["FLAG_SEVERE_DELECENCY"] = (pos["SK_DPD"] > 90)
pos["FULLY_PAID_RATIO"] = 1 - (pos["CNT_INSTALMENT_FUTURE"] / pos["CNT_INSTALMENT"])
pos["FLAG_COMPLETED"] = pos["NAME_CONTRACT_STATUS"] == "Completed"

new_df = pos.groupby("SK_ID_PREV").agg({
    "FLAG_SEVERE_DELECENCY": ["count"],
    "FULLY_PAID_RATIO": ["mean"],
    "FLAG_COMPLETED": ["count"],
    "SK_DPD": ["mean"],
    
})

new_df.columns = ["POS_FLAG_SEVERE_DELECENCY_COUNT", "POS_FULLY_PAID_RATIO_MEAN", "POS_FLAG_COMPLETED_COUNT", "POS_DPD_MEAN"]

In [35]:
print(new_df.shape)
new_df.isna().sum()

(936325, 4)


POS_FLAG_SEVERE_DELECENCY_COUNT      0
POS_FULLY_PAID_RATIO_MEAN          890
POS_FLAG_COMPLETED_COUNT             0
POS_DPD_MEAN                         0
dtype: int64

- push to db
- install spark
- 